# 11 · Rendimiento y escalabilidad

**Propósito:** medir, con datos reales, (1) cuánto mejoran cuatro optimizaciones de Spark y (2) cómo crece el tiempo
del pipeline de silver a gold cuando crece el volumen, y desde qué punto una sola máquina con pandas deja de
alcanzar.

**Entrada:** `silver/vuelos`, `silver/clima_horario` y la ficha de aeropuertos.

**Salida:** `docs/evidencias/rendimiento/optimizaciones.csv`, `escalabilidad.csv` (se le agregan filas en cada
corrida) y `escalabilidad.png`. Las réplicas de la prueba se escriben **solo** en `scale-test/`, que el bucket borra
a los 7 días y que nunca se usa en el análisis ni en el modelo.

**Requisito:** notebooks 05 y 06.

### Cómo repetir la prueba con más nodos

1. En `infra/procesamiento/terraform.tfvars` poner `worker_count = 0`, `1` o `2` y hacer `terraform apply`.
2. Cambiar `NODOS` en la celda siguiente (nodos con worker = `worker_count + 1`, porque el master también procesa).
3. Ejecutar el notebook completo. Cada corrida **agrega** filas a los CSV con su número de nodos, y el gráfico las
   compara.

Las métricas de cada etapa salen de la API REST de Spark (`/api/v1/applications/<id>/stages`): tiempo de ejecución en
los executors, bytes derramados a memoria y a disco (*spill*) y bytes leídos en el *shuffle*.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import resource
import time
from datetime import UTC, datetime

import pandas as pd
import requests

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-rendimiento")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Paleta: una serie principal en azul; segunda y tercera en naranja y aqua; lo que no se destaca, en gris
AZUL, NARANJA, AQUA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c2"
TINTA, TINTA2, REJILLA, FONDO = "#0b0b0b", "#52514e", "#e7e6e1", "#fcfcfb"
matplotlib.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": REJILLA, "axes.labelcolor": TINTA2, "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "figure.dpi": 110, "legend.frameon": False,
})

In [ ]:
NODOS = 1  # nodos con worker en esta corrida: worker_count + 1

EVID = RAIZ / "docs" / "evidencias" / "rendimiento"
EVID.mkdir(parents=True, exist_ok=True)
ANIO_BASE = cfg["escalabilidad"]["anio_base"]
REPLICAS = cfg["escalabilidad"]["replicas"]
UMBRAL_AMBAR = cfg["umbrales_demora_min"]["ambar"]
UI = spark.sparkContext.uiWebUrl
APP = spark.sparkContext.applicationId
print(f"Spark UI: {UI} · aplicación {APP} · nodos declarados: {NODOS}")


def etapas():
    respuesta = requests.get(f"{UI}/api/v1/applications/{APP}/stages", timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


def medir(accion):
    """Ejecuta la acción y devuelve el tiempo de pared y la suma de las métricas de las etapas nuevas."""
    previas = {(e["stageId"], e["attemptId"]) for e in etapas()}
    inicio = time.perf_counter()
    accion()
    segundos = time.perf_counter() - inicio
    time.sleep(2)  # la API publica las etapas terminadas con un pequeño retraso
    nuevas = [e for e in etapas() if (e["stageId"], e["attemptId"]) not in previas and e["status"] == "COMPLETE"]
    suma = lambda clave: sum(e.get(clave, 0) for e in nuevas)  # noqa: E731
    return {
        "segundos": round(segundos, 2), "etapas": len(nuevas),
        "executor_run_s": round(suma("executorRunTime") / 1000, 2),
        "entrada_mb": round(suma("inputBytes") / 1e6, 1),
        "shuffle_lectura_mb": round(suma("shuffleReadBytes") / 1e6, 1),
        "spill_memoria_mb": round(suma("memoryBytesSpilled") / 1e6, 1),
        "spill_disco_mb": round(suma("diskBytesSpilled") / 1e6, 1),
    }


def sin_salida(df):
    """Fuerza el cálculo completo sin escribir nada (formato noop de Spark)."""
    df.write.format("noop").mode("overwrite").save()


def agregar_csv(filas, archivo):
    nuevo = pd.DataFrame(filas)
    ruta_csv = EVID / archivo
    if ruta_csv.exists():
        nuevo = pd.concat([pd.read_csv(ruta_csv), nuevo], ignore_index=True)
    nuevo.to_csv(ruta_csv, index=False)
    return nuevo


vuelos = en_periodo(spark.read.parquet(ruta("silver_vuelos")))
clima_horario = spark.read.parquet(ruta("silver_clima_horario"))
esquema_aeropuertos = T.StructType([
    T.StructField("aeropuerto", T.StringType()), T.StructField("estacion", T.StringType()),
    T.StructField("nombre", T.StringType()), T.StructField("pais", T.StringType()),
    T.StructField("latitud", T.DoubleType()), T.StructField("longitud", T.DoubleType()),
    T.StructField("zona_horaria", T.StringType()), T.StructField("tiene_metar", T.BooleanType()),
])
aeropuertos = spark.read.option("header", True).schema(esquema_aeropuertos).csv(ruta("bronze_aeropuertos"))
ahora = datetime.now(UTC).isoformat(timespec="seconds")
optimizaciones = []

## 1. Optimizaciones

### O1. Broadcast frente a sort-merge en un join con una tabla chica

Unir los vuelos con la ficha de aeropuertos (unos cientos de filas). Con *broadcast* la tabla chica se copia a cada
nodo y los vuelos no se mueven; sin él, Spark reparte y ordena **los dos lados** por la clave (*shuffle*).

In [ ]:
umbral_original = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")


def o1(con_broadcast):
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral_original if con_broadcast else "-1")
    tabla = F.broadcast(aeropuertos) if con_broadcast else aeropuertos
    sin_salida(vuelos.join(tabla, vuelos.origen == tabla.aeropuerto, "left")
               .groupBy("zona_horaria").agg(F.avg("demora_salida_min")))


for variante, flag in [("sort-merge (antes)", False), ("broadcast (después)", True)]:
    optimizaciones.append({"optimizacion": "O1 broadcast join", "variante": variante, **medir(lambda: o1(flag))})
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral_original)
pd.DataFrame(optimizaciones)

### O2. Poda de particiones y de columnas

Leer un mes de LATAM Brasil. **Antes:** se filtra por una columna calculada (Spark no puede saltarse carpetas) y se
leen todas las columnas. **Después:** se filtra por las columnas de partición (`pais`, `year`, `month`), así Spark
solo abre las carpetas de ese mes, y se leen únicamente las columnas necesarias (Parquet es columnar).

In [ ]:
anio_o2, mes_o2 = map(int, HASTA.split("-"))


def o2_completa():
    df = spark.read.parquet(ruta("silver_vuelos"))
    sin_salida(df.where((F.date_format("fecha_local", "yyyy-MM") == HASTA) & (F.col("es_latam"))))


def o2_poda():
    df = spark.read.parquet(ruta("silver_vuelos"))
    sin_salida(df.where((F.col("pais") == "BR") & (F.col("year") == anio_o2) & (F.col("month") == mes_o2))
               .where(F.col("es_latam")).select("origen", "salida_prog_utc", "demora_salida_min"))


optimizaciones.append({"optimizacion": "O2 poda de particiones y columnas", "variante": "lectura completa (antes)",
                       **medir(o2_completa)})
optimizaciones.append({"optimizacion": "O2 poda de particiones y columnas", "variante": "con poda (después)",
                       **medir(o2_poda)})
spark.read.parquet(ruta("silver_vuelos")).where(
    (F.col("pais") == "BR") & (F.col("year") == anio_o2) & (F.col("month") == mes_o2)
).select("origen").explain()

### O3. Cache de un DataFrame que se usa varias veces

Tres agregaciones sobre los vuelos operados. **Antes:** cada una vuelve a leer y filtrar desde el lake.
**Después:** el resultado del filtro se guarda en memoria con `cache()` y las tres lo reutilizan.

In [ ]:
def o3(con_cache):
    operados = vuelos.where(~F.col("cancelado") & F.col("demora_salida_min").isNotNull()).select(
        "pais", "origen", "aerolinea", "hora_local_salida", "y_demora15")
    if con_cache:
        operados = operados.cache()
        operados.count()
    sin_salida(operados.groupBy("pais", "origen").agg(F.avg("y_demora15")))
    sin_salida(operados.groupBy("pais", "aerolinea").agg(F.avg("y_demora15")))
    sin_salida(operados.groupBy("pais", "hora_local_salida").agg(F.avg("y_demora15")))
    if con_cache:
        operados.unpersist()


optimizaciones.append({"optimizacion": "O3 cache", "variante": "sin cache (antes)", **medir(lambda: o3(False))})
optimizaciones.append({"optimizacion": "O3 cache", "variante": "con cache (después)", **medir(lambda: o3(True))})

### O4. Repartition antes de una ventana

Una ventana por aeropuerto de origen (la demora de la salida anterior del mismo aeropuerto). **Antes:** Spark decide
el reparto. **Después:** se reparte explícitamente por la clave de la ventana, `(pais, origen)`, antes de calcularla.

In [ ]:
ventana = Window.partitionBy("pais", "origen").orderBy("salida_prog_utc")


def o4(con_repartition):
    df = vuelos.select("pais", "origen", "salida_prog_utc", "demora_salida_min")
    if con_repartition:
        df = df.repartition("pais", "origen")
    sin_salida(df.withColumn("demora_anterior", F.lag("demora_salida_min").over(ventana)))


optimizaciones.append({"optimizacion": "O4 repartition antes de la ventana", "variante": "sin repartition (antes)",
                       **medir(lambda: o4(False))})
optimizaciones.append({"optimizacion": "O4 repartition antes de la ventana", "variante": "con repartition (después)",
                       **medir(lambda: o4(True))})

In [ ]:
tabla_opt = pd.DataFrame(optimizaciones)
tabla_opt.insert(0, "nodos", NODOS)
tabla_opt.insert(0, "fecha_utc", ahora)
agregar_csv(tabla_opt.to_dict("records"), "optimizaciones.csv")
tabla_opt

## 2. Escalabilidad del pipeline de silver a gold

Se mide el cálculo más pesado de gold sobre los vuelos: la unión con el clima del origen, el estado de la red en las
dos horas previas, la congestión programada y la demora del mismo vuelo el día anterior. El resultado no se escribe
(formato `noop`): se mide el cálculo, no la escritura.

| Escalón | Datos |
|---|---|
| E1 | 1 mes real |
| E2 | 1 año real (`escalabilidad.anio_base`) |
| E3 | el periodo completo real (Brasil + EE. UU.) |
| E4 | réplica `escalabilidad.replicas` veces del año base, escrita en `scale-test/` |
| E5 | el mismo cálculo con pandas en una sola máquina, sobre volúmenes crecientes |

In [ ]:
def calculo_gold(v):
    """Las mismas uniones y agregaciones del notebook 08, sobre el DataFrame de vuelos que se le pase."""
    base = (v.where(~F.col("cancelado") & ~F.col("desviado") & ~F.col("es_atipico") & F.col("y_demora15").isNotNull())
            .withColumn("k_ref", F.date_trunc("hour", F.col("salida_prog_utc") - F.expr("INTERVAL 2 HOURS"))))
    clima_o = clima_horario.select(F.col("aeropuerto").alias("origen"), F.col("k").alias("k_ref"),
                                   F.col("visibilidad_m").alias("o_vis"), F.col("techo_ft").alias("o_techo"))
    despegados = v.where(F.col("salida_real_utc").isNotNull()).select(
        "pais", "origen", F.date_trunc("hour", "salida_prog_utc").alias("hora_prog"), "salida_real_utc",
        "demora_salida_min")
    aportes = None
    for horas in (1, 2):
        parte = (despegados.withColumn("k_ref", F.col("hora_prog") + F.expr(f"INTERVAL {horas} HOURS"))
                 .where(F.col("salida_real_utc") < F.col("k_ref")))
        aportes = parte if aportes is None else aportes.unionByName(parte)
    red = aportes.groupBy("pais", "origen", "k_ref").agg(
        F.count("*").alias("red_n"), F.avg((F.col("demora_salida_min") >= UMBRAL_AMBAR).cast("int")).alias("red_frac"))
    congestion = v.groupBy("pais", "origen", "fecha_local", "hora_local_salida").agg(F.count("*").alias("cong"))
    ayer = (v.where(F.col("demora_salida_min").isNotNull())
            .withColumn("fecha_local", F.date_add("fecha_local", 1))
            .groupBy("pais", "aerolinea", "numero_vuelo", "origen", "fecha_local")
            .agg(F.avg("demora_salida_min").alias("prev")))
    return (base.join(clima_o, ["origen", "k_ref"], "left")
            .join(red, ["pais", "origen", "k_ref"], "left")
            .join(congestion, ["pais", "origen", "fecha_local", "hora_local_salida"], "left")
            .join(ayer, ["pais", "aerolinea", "numero_vuelo", "origen", "fecha_local"], "left"))


escalones = []


def correr_spark(escalon, df, nota=""):
    filas = df.count()
    try:
        resultado = medir(lambda: sin_salida(calculo_gold(df)))
        exito = True
    except Exception as error:  # se registra el fallo (por ejemplo, falta de memoria) como resultado de la prueba
        resultado, exito, nota = {}, False, f"{type(error).__name__}: {str(error)[:150]}"
    escalones.append({"fecha_utc": ahora, "escalon": escalon, "motor": "spark", "nodos": NODOS, "filas": filas,
                      **resultado, "exito": exito, "nota": nota})
    print(escalones[-1])


anio_fin, mes_fin = map(int, HASTA.split("-"))
correr_spark("E1 1 mes", vuelos.where((F.col("year") == anio_fin) & (F.col("month") == mes_fin)))
correr_spark(f"E2 año {ANIO_BASE}", vuelos.where(F.col("year") == ANIO_BASE))
correr_spark("E3 periodo completo", vuelos)

### E4. Réplica del año base

Se copia el año base `replicas` veces. Cada copia se desplaza 364 días por bloque (así conserva el día de la semana)
y su número de vuelo lleva el sufijo `-R<n>`, para que las copias no se mezclen al buscar el vuelo del día anterior.
La réplica se escribe en `scale-test/`, se mide y se deja para que el bucket la borre a los 7 días.

In [ ]:
anio_base = vuelos.where(F.col("year") == ANIO_BASE)
copias = None
for n in range(REPLICAS):
    segundos = n * 364 * 86400
    copia = anio_base
    for c in ["salida_prog_utc", "salida_real_utc", "llegada_prog_utc", "llegada_real_utc"]:
        copia = copia.withColumn(c, F.timestamp_seconds(F.unix_timestamp(c) + segundos))
    copia = (copia.withColumn("fecha_local", F.date_add("fecha_local", n * 364))
             .withColumn("numero_vuelo", F.concat("numero_vuelo", F.lit(f"-R{n}")))
             .withColumn("year", F.year("fecha_local")).withColumn("month", F.month("fecha_local")))
    copias = copia if copias is None else copias.unionByName(copia)
(
    copias.repartition("pais", "year", "month")
    .write.mode("overwrite").partitionBy("pais", "year", "month").parquet(ruta("scale_test"))
)
correr_spark(f"E4 réplica {REPLICAS}x del año {ANIO_BASE}", spark.read.parquet(ruta("scale_test")),
             "datos replicados en scale-test/, solo para esta prueba")

### E5. Una sola máquina con pandas

El mismo cálculo (red, congestión y día anterior) con pandas, leyendo el Parquet directamente con `pyarrow`. Se
prueba con 1 mes, el año base y réplicas crecientes del año base. Antes de cada escalón se estima la memoria
necesaria; si supera la mitad de la memoria disponible de la máquina, **no se ejecuta** (el sistema mataría el
proceso) y se registra como el punto donde una sola máquina deja de alcanzar.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.fs as pafs

COLUMNAS_PD = ["pais", "aerolinea", "numero_vuelo", "origen", "salida_prog_utc", "salida_real_utc",
               "fecha_local", "hora_local_salida", "demora_salida_min"]
if BUCKET:
    sistema = pafs.S3FileSystem(region=os.environ.get("AWS_REGION", "us-east-1"))
    ubicacion = f"{BUCKET}/{tablas['silver_vuelos']}"
else:
    sistema = pafs.LocalFileSystem()
    ubicacion = str((RAIZ_LOCAL / tablas["silver_vuelos"]).resolve())
dataset = ds.dataset(ubicacion, filesystem=sistema, format="parquet", partitioning="hive")


def memoria_disponible_gb():
    with open("/proc/meminfo") as f:
        datos = dict(linea.split(":", 1) for linea in f)
    return int(datos["MemAvailable"].split()[0]) / 1e6


def calculo_pandas(v):
    v = v.copy()
    v["k_ref"] = (v["salida_prog_utc"] - pd.Timedelta(hours=2)).dt.floor("h")
    v["hora_prog"] = v["salida_prog_utc"].dt.floor("h")
    despegados = v[v["salida_real_utc"].notna()]
    partes = []
    for horas in (1, 2):
        p = despegados.assign(k_ref=despegados["hora_prog"] + pd.Timedelta(hours=horas))
        partes.append(p[p["salida_real_utc"] < p["k_ref"]])
    aportes = pd.concat(partes)
    aportes["atrasada"] = aportes["demora_salida_min"] >= UMBRAL_AMBAR
    red = aportes.groupby(["pais", "origen", "k_ref"]).agg(red_n=("atrasada", "size"), red_frac=("atrasada", "mean"))
    v = v.merge(red.reset_index(), on=["pais", "origen", "k_ref"], how="left")
    v["cong"] = v.groupby(["pais", "origen", "fecha_local", "hora_local_salida"])["pais"].transform("size")
    ayer = v[["pais", "aerolinea", "numero_vuelo", "origen", "fecha_local", "demora_salida_min"]].copy()
    ayer["fecha_local"] = ayer["fecha_local"] + pd.Timedelta(days=1)
    ayer = ayer.groupby(["pais", "aerolinea", "numero_vuelo", "origen", "fecha_local"], as_index=False)[
        "demora_salida_min"].mean().rename(columns={"demora_salida_min": "prev"})
    return v.merge(ayer, on=["pais", "aerolinea", "numero_vuelo", "origen", "fecha_local"], how="left")


def correr_pandas(escalon, filtro, veces=1):
    base_pd = dataset.to_table(columns=COLUMNAS_PD, filter=filtro).to_pandas()
    base_pd["fecha_local"] = pd.to_datetime(base_pd["fecha_local"])
    gb_base = base_pd.memory_usage(deep=True).sum() / 1e9
    necesaria = gb_base * veces * 4  # datos + copias intermedias de los merge y groupby
    disponible = memoria_disponible_gb()
    fila = {"fecha_utc": ahora, "escalon": escalon, "motor": "pandas", "nodos": 1, "filas": len(base_pd) * veces,
            "memoria_estimada_gb": round(necesaria, 2), "memoria_disponible_gb": round(disponible, 2)}
    if necesaria > disponible * 0.5:
        escalones.append({**fila, "exito": False,
                          "nota": "no se ejecuta: la memoria estimada supera la mitad de la disponible"})
        print(escalones[-1])
        return False
    datos_pd = pd.concat(
        [base_pd.assign(numero_vuelo=base_pd["numero_vuelo"] + f"-R{n}",
                        salida_prog_utc=base_pd["salida_prog_utc"] + pd.Timedelta(days=364 * n),
                        salida_real_utc=base_pd["salida_real_utc"] + pd.Timedelta(days=364 * n),
                        fecha_local=base_pd["fecha_local"] + pd.Timedelta(days=364 * n))
         for n in range(veces)], ignore_index=True) if veces > 1 else base_pd
    inicio = time.perf_counter()
    try:
        calculo_pandas(datos_pd)
        exito, nota = True, ""
    except MemoryError as error:
        exito, nota = False, f"MemoryError: {error}"
    fila.update({"segundos": round(time.perf_counter() - inicio, 2),
                 "pico_rss_gb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2),
                 "exito": exito, "nota": nota})
    escalones.append(fila)
    print(escalones[-1])
    del datos_pd, base_pd
    return exito


correr_pandas("E5 pandas 1 mes", (ds.field("year") == anio_fin) & (ds.field("month") == mes_fin))
for veces in [1, 2, 5, REPLICAS]:
    etiqueta = f"E5 pandas año {ANIO_BASE}" + (f" x{veces}" if veces > 1 else "")
    if not correr_pandas(etiqueta, ds.field("year") == ANIO_BASE, veces):
        break

## 3. Resultados

In [ ]:
from matplotlib.ticker import FuncFormatter, NullFormatter

historico = agregar_csv(escalones, "escalabilidad.csv")
display(pd.DataFrame(escalones))

ok = historico[historico["exito"] == True]  # noqa: E712
fig, ax = plt.subplots(figsize=(9.5, 5.4))
fig.subplots_adjust(top=0.8, bottom=0.14, left=0.1, right=0.97)
series = [(f"Spark · {n} nodo(s)", (ok["motor"] == "spark") & (ok["nodos"] == n)) for n in sorted(ok[ok["motor"] == "spark"]["nodos"].unique())]
series.append(("pandas · 1 máquina", ok["motor"] == "pandas"))
for (nombre, filtro), color in zip(series, [AZUL, NARANJA, AQUA, GRIS]):
    d = ok[filtro].sort_values("filas")
    if len(d):
        ax.plot(d["filas"], d["segundos"], marker="o", ms=6, lw=2, color=color, label=nombre)
fallos = historico[historico["exito"] == False]  # noqa: E712
for _, f in fallos.iterrows():
    ax.axvline(f["filas"], color=TINTA2, lw=1, ls="--")
    ax.text(f["filas"], ax.get_ylim()[1] if len(ok) else 1, f" {f['motor']}: no alcanza", fontsize=8, color=TINTA2,
            rotation=90, va="top")
ax.set_xscale("log")
ax.set_yscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1e6:g} M"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g} s"))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("Vuelos procesados (escala logarítmica)")
ax.set_ylabel("Segundos (escala logarítmica)")
ax.legend(loc="upper left")
fig.text(0.02, 0.95, "¿Cómo crece el tiempo del pipeline con el volumen?", fontsize=13, fontweight="bold", color=TINTA)
fig.text(0.02, 0.89, "Tiempo del cálculo de silver a gold por escalón. Las líneas punteadas marcan dónde un motor "
         "deja de alcanzar.", fontsize=10, color=TINTA2)
fig.savefig(EVID / "escalabilidad.png", dpi=160)
plt.show()

In [ ]:
spark.stop()